In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier, XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [3]:

# 1.0 CONSTANTS & DATA -----
profit_margin = 0.15 
transactions_df = pd.read_csv('data_transactions_processed.csv')
df = transactions_df.copy()
df['timestamp'] = pd.to_datetime(df['timestamp'])


In [ ]:

# 2.0 TARGET DEFINITION (90-Day Window) -----
n_days = 90
max_date = df['timestamp'].max()
cutoff = max_date - pd.to_timedelta(n_days, unit="d")

temporal_in_df  = df[df['timestamp'] < cutoff]
temporal_out_df = df[df['timestamp'] >= cutoff] \
    .query('household_key in @temporal_in_df.household_key')

targets_df = temporal_out_df[['household_key', 'timestamp', 'sales_value']] \
    .groupby('household_key') \
    .sum() \
    .rename({'sales_value': 'sales_90_value'}, axis=1) \
    .assign(sales_90_flag = 1)

# 3.0 FEATURE ENGINEERING -----
max_date_in = temporal_in_df['timestamp'].max()

# Recency
recency_df = temporal_in_df.groupby('household_key')['timestamp'] \
    .apply(lambda x: (x.max() - max_date_in).days) \
    .to_frame(name='recency')

# Frequency
freq_df = temporal_in_df.groupby('household_key').size().to_frame(name='frequency')

# Monetary
mon_df = temporal_in_df.groupby('household_key')['sales_value'] \
    .agg(['sum', 'mean']).rename(columns={'sum': 'sales_value_sum', 'mean': 'sales_value_mean'})

# Time-based Windows
cutoff_28d = cutoff - pd.to_timedelta(28, unit="d")
cutoff_14d = cutoff - pd.to_timedelta(14, unit="d")

tx_28d = temporal_in_df.query('timestamp >= @cutoff_28d').groupby('household_key').size().to_frame('tx_last_month')
tx_14d = temporal_in_df.query('timestamp >= @cutoff_14d').groupby('household_key').size().to_frame('tx_last_2w')
sales_14d = temporal_in_df.query('timestamp >= @cutoff_14d').groupby('household_key')['sales_value'].sum().to_frame('sales_last_2w')

# Combine
features_df = pd.concat([recency_df, freq_df, mon_df, tx_28d, tx_14d, sales_14d], axis=1) \
    .merge(targets_df, left_index=True, right_index=True, how="left") \
    .fillna(0)

# 4.0 MANUAL MACHINE LEARNING -----
X = features_df.drop(['sales_90_value', 'sales_90_flag'], axis=1)
y_reg = features_df['sales_90_value']
y_clf = features_df['sales_90_flag']

# Scaling (Manual Normalization)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

# Regression Model
xgb_reg = XGBRegressor(n_estimators=100, random_state=123).fit(X_scaled, y_reg)
features_df['prediction_label'] = xgb_reg.predict(X_scaled)

# Classification Model
xgb_clf = XGBClassifier(n_estimators=100, random_state=123).fit(X_scaled, y_clf)
features_df['prediction_score_1'] = xgb_clf.predict_proba(X_scaled)[:, 1]

# 5.0 BUSINESS VALUE -----
top_20_customers = features_df.sort_values('prediction_label', ascending=False).head(20).index.tolist()

# Increase Frequency Insights
freq_ins = transactions_df.query('household_key in @top_20_customers') \
    .groupby('commodity_desc').size().sort_values(ascending=False)

# Increase Size Insights
size_ins = transactions_df.query('household_key in @top_20_customers') \
    .groupby('commodity_desc')['sales_value'].sum().sort_values(ascending=False)

print("Top 5 Commodities by Frequency:\n", freq_ins.head(5))